# D06 · CAN at the bit level - arbitration, stuffing, timing, load

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/02_drivers_protocols/D06_can_at_the_bit_level_arbitration_stuffing_timing.ipynb)

**Track:** Drivers and protocols · **Skill axis (sheet):** Drivers and protocols · **Time:** about 65 min

**Skill Hub concepts:** CAN arbitration; CAN bit stuffing; CAN bit timing; CAN bus load

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("D06")

## Map: one classic CAN data frame (11-bit identifier)

```
| SOF | ID[10:0] | RTR | IDE | r0 | DLC[3:0] | DATA 0..64 bits | CRC 15 | CRC del | ACK slot | ACK del | EOF 7 | IFS 3 |
|  1  |    11    |  1  |  1  |  1 |    4     |      8n         |   15   |    1    |    1     |    1    |   7   |   3   |
 <------------------------------- bit-stuffed region: 34 + 8n bits -------------------------------->
```

| Rule | Detail |
|---|---|
| Dominant 0 beats recessive 1 | wired-AND on CANH/CANL: lowest identifier wins arbitration, losers become receivers, no frame is destroyed |
| Bit stuffing | after 5 equal bits the sender inserts one opposite bit (SOF to end of CRC); guarantees edges for resync |
| Bit time | $1\ \text{bit} = (1 + \text{TSEG1} + \text{TSEG2})\,t_q$, $t_q = \text{BRP}/f_{clk}$, TSEG1 = PROP_SEG + PHASE_SEG1 |
| Sample point | $(1 + \text{TSEG1})/(1 + \text{TSEG1} + \text{TSEG2})$, about 87.5% for CANopen and 80% common |
| Worst-case frame | $C = 8n + 47 + \left\lfloor \frac{34 + 8n - 1}{4} \right\rfloor$ bits (Davis et al.), 135 bits for 8 bytes |

Bus load $= \sum_i C_i \, r_i \,/\, \text{bitrate}$ with $r_i$ in frames per second.

## Task 1: arbitration
Return `(winner_id, {loser_id: bit_index_where_it_lost})` for 11-bit identifiers sent MSB first (index 0 = ID bit 10).
Simulate the wired-AND bit by bit.

In [ ]:
def arbitrate(ids):
    raise NotImplementedError  # TODO

lab.check("arbitrate", arbitrate)

In [ ]:
%%writefile d06_arbitrate.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/d06.h
// Grade it with lab.check("arbitrate", "d06_arbitrate.c"). For C++, name the file .cpp in both places.
#include "d06.h"

uint16_t can_arbitrate(const uint16_t *ids, size_t n, int *lost_at) {
    for (size_t i = 0; i < n; ++i) lost_at[i] = -1;
    (void)ids;
    return 0; /* TODO: bit by bit from the MSB; a node sending 1 while the wire is 0 drops out */
}

In [ ]:
# lab.check("arbitrate", "d06_arbitrate.c")  # uncomment after writing the C/C++ cell above

## Task 2: stuff and unstuff

In [ ]:
def stuff(bits):
    raise NotImplementedError  # TODO

def unstuff(bits):
    raise NotImplementedError  # TODO

lab.check("stuffing", stuff, unstuff)

In [ ]:
%%writefile d06_stuffing.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/d06.h
// Grade it with lab.check("stuffing", "d06_stuffing.c"). For C++, name the file .cpp in both places.
#include "d06.h"

size_t can_stuff(const uint8_t *bits, size_t n, uint8_t *out, size_t out_cap) {
    (void)bits; (void)n; (void)out; (void)out_cap;
    return 0; /* TODO: after 5 equal bits insert the complement */
}

size_t can_unstuff(const uint8_t *bits, size_t n, uint8_t *out, size_t out_cap) {
    (void)bits; (void)n; (void)out; (void)out_cap;
    return 0; /* TODO */
}

In [ ]:
# lab.check("stuffing", "d06_stuffing.c")  # uncomment after writing the C/C++ cell above

## Task 3: bit timing and bus load

In [ ]:
def bit_timing(f_clk, bitrate, sample_point=0.8):
    """Return (brp, tseg1, tseg2) with 8..25 time quanta per bit, tseg2 >= 1, exact bitrate,
    sample point as close as possible to the target (prefer more quanta per bit on a tie)."""
    raise NotImplementedError  # TODO

def frame_bits_worst(n_bytes):
    raise NotImplementedError  # TODO

def bus_load(messages, bitrate):
    """messages: list of (n_bytes, rate_hz). Worst-case load as a fraction."""
    raise NotImplementedError  # TODO

lab.check("timing_load", bit_timing, frame_bits_worst, bus_load)

In [ ]:
%%writefile d06_timing.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/d06.h
// Grade it with lab.check("timing_load", "d06_timing.c"). For C++, name the file .cpp in both places.
#include "d06.h"

void can_bit_timing(uint32_t f_clk, uint32_t bitrate, double sample_point, uint32_t *brp, uint32_t *tseg1, uint32_t *tseg2) {
    (void)f_clk; (void)bitrate; (void)sample_point;
    *brp = 0; *tseg1 = 0; *tseg2 = 0; /* TODO */
}

uint32_t can_frame_bits_worst(uint32_t n_bytes) { (void)n_bytes; return 0; /* TODO */ }

double can_bus_load(const uint32_t *n_bytes, const double *rate_hz, size_t n_messages, uint32_t bitrate) {
    (void)n_bytes; (void)rate_hz; (void)n_messages; (void)bitrate;
    return 0; /* TODO */
}

In [ ]:
# lab.check("timing_load", "d06_timing.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** Worst-case bits of a standard frame with 8 data bytes?

**M2 (Apply).** 170 MHz, BRP = 17, 500 kbit/s, 80% sample point. TSEG2 in time quanta?

**M3 (Analyze).** Ten 8-byte messages, each at 100 Hz, on 500 kbit/s. Worst-case bus load as a fraction?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** Two nodes transmit identifier 0x120 at the same instant with different data. What happens?
- a) the lower data value wins
- b) arbitration cannot separate them; they collide in the data field, a bit error follows, both send error frames
- c) both frames arrive intact

**K2 (Understand).** Explain why stuffing exists, why the bus length limits the bit rate, and why the sample point
sits late in the bit.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `d06.h`).
3. In `roboprep/checks/d06.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()